# Colab용 영화 리뷰 분석기 (Ollama - A100/gemma4 최적화)
이 노트북은 구글 드라이브를 마운트하고 Ollama를 로컬에 설치한 뒤, 리뷰 데이터를 분석하는 과정을 담고 있습니다.

In [1]:
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'
if os.path.exists(mountpoint) and not os.path.ismount(mountpoint):
    shutil.rmtree(mountpoint)

drive.mount(mountpoint, force_remount=True)

Mounted at /content/drive


In [2]:
# 2. Ollama 설치 및 백그라운드 실행, 모델 다운로드
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

!pip install ollama

# A100 GPU 환경에 맞춘 gemma4 모델 다운로드
!ollama pull gemma4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [3]:
import subprocess
try:
    gpu_status = subprocess.check_output(['nvidia-smi']).decode('utf-8')
    print("✅ GPU가 감지되었습니다:")
    print(gpu_status)
except Exception:
    print("❌ GPU가 감지되지 않았습니다. 런타임 유형이 GPU로 설정되었는지 확인해 주세요.")

✅ GPU가 감지되었습니다:
Fri May  1 12:44:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [4]:
# 3. 라이브러리 임포트 및 경로 설정
import json
import os
import time
import re
import ollama

ROOT_DIR = "/content/drive/MyDrive/MovieReview/"
DATA_DIR = os.path.join(ROOT_DIR, "data")

LOCAL_MODEL = "gemma4"

IS_TEST_MODE = False
TEST_LIMIT = 100

print(f"데이터 경로 설정 완료: {DATA_DIR}")
print(f"사용 모델 설정 완료: {LOCAL_MODEL}")

데이터 경로 설정 완료: /content/drive/MyDrive/MovieReview/data
사용 모델 설정 완료: gemma4


In [5]:
# 4. 태그 룰 & 분류 함수 정의
TAG_RULES = {
    "연기좋음": ["열연", "명연기", "연기력", "인생 캐릭터", "소름", "연기 잘"],
    "연기나쁨": ["연기력 논란", "발연기", "어색", "몰입 방해", "연기 못"],
    "연출좋음": ["영상미", "미장센", "감각적", "탁월한 연출", "연출력"],
    "연출나쁨": ["조잡", "촌스러운", "연출력 부족", "엉성한 연출"],
    "서사좋음": ["탄탄한", "짜임새", "명작", "완벽한 서사", "개연성"],
    "서사나쁨": ["지루", "뻔한", "용두사미", "허술", "개연성 부족"],
    "비주얼좋음": ["비주얼", "영상미", "눈이 즐거운", "그래픽", "훌륭한 CG"],
    "비주얼나쁨": ["허접한 CG", "촌스러운 비주얼", "어색한 그래픽"],
    "음악좋음": ["ost", "음악", "사운드트랙", "배경음악", "음악이 좋"],
    "음악나쁨": ["음악이 안 어울", "시끄러운", "음악이 별로"],
    "분위기좋음": ["분위기 있", "감성적인", "매력적인 분위기", "무드"],
    "분위기나쁨": ["산만한 분위기", "분위기 깨는", "칙칙한"],
    "주의사항": ["노출", "가족", "민망", "야한", "잔인", "공포", "폭력"],
    "전체긍정": ["최고", "추천", "재밌", "훌륭", "완벽", "인생 영화", "강추"],
    "전체부정": ["최악", "비추", "노잼", "별로", "시간 아깝", "실망", "돈 아깝"]
}

def classify_with_rule_base(content):
    found_tags = []
    extracted_keywords = []
    for tag_name, keywords in TAG_RULES.items():
        matched = [k for k in keywords if k in content]
        if matched:
            found_tags.append(tag_name)
            extracted_keywords.extend(matched)
    if not found_tags: found_tags.append("일반감상")
    return found_tags, list(set(extracted_keywords))

def classify_single_review_with_local_llm(review, max_retries=5):
    prompt = f"""
    당신은 영화 리뷰 분석 전문가입니다. 아래 리뷰를 읽고 범주에서 알맞은 태그를 선택하고, 핵심 키워드를 추출하세요.

    [범주]: 전체긍정, 전체부정, 전체복합, 연기좋음, 연기나쁨, 연출좋음, 연출나쁨, 서사좋음, 서사나쁨, 비주얼좋음, 비주얼나쁨, 음악좋음, 분위기가벼움, 분위기무거움, 고증좋음, 고증나쁨, 주의사항, TMI, 장르특성

    [지시사항]
    - 반드시 아래 JSON 형식으로만 대답하세요. 다른 인사말이나 설명은 절대 넣지 마세요.
    - content_character 에는 범주에 해당하는 태그만 넣으세요.
    - search_keywords 에는 리뷰 본문에서 추출한 핵심 단어를 넣으세요.

    {{
        "content_character": ["태그1", "태그2"],
        "search_keywords": ["키워드1", "키워드2"]
    }}

    [리뷰 본문]:
    "{review['content']}"
    """

    import time
    for attempt in range(max_retries):
        try:
            response = ollama.chat(model=LOCAL_MODEL, messages=[
                {'role': 'user', 'content': prompt},
            ], format='json')

            response_text = response['message']['content']
            json_text = re.sub(r'```json|```', '', response_text).strip()
            res = json.loads(json_text)

            content_chars = res.get("content_character", [])
            keywords = res.get("search_keywords", [])

            if not content_chars or not keywords:
                raise ValueError("태그나 키워드가 비어있습니다.")

            allowed_tags = {"전체긍정", "전체부정", "전체복합", "연기좋음", "연기나쁨", "연출좋음", "연출나쁨", "서사좋음", "서사나쁨", "비주얼좋음", "비주얼나쁨", "음악좋음", "분위기가벼움", "분위기무거움", "고증좋음", "고증나쁨", "주의사항", "TMI", "장르특성"}
            res["content_character"] = [t for t in content_chars if t in allowed_tags]
            if not res["content_character"]:
                res["content_character"] = ["일반감상"]

            return res
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(0.5)
            else:
                print(f"⚠️ 분석 실패 (Fallback 전환): {e}")
                return None

In [6]:
# 5. 메인 실행 블록
from tqdm.notebook import tqdm
import os
import json

input_path = os.path.join(DATA_DIR, "cleaned_total_reviews.json")

if IS_TEST_MODE:
    output_path = os.path.join(DATA_DIR, "test_tagged_reviews.json")
else:
    output_path = os.path.join(DATA_DIR, "tagged_reviews.json")

if not os.path.exists(input_path):
    print(f"❌ '{input_path}' 파일이 없습니다. 경로를 확인해 주세요.")
else:
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
        print("📝 기존 작업 결과에서 이어하기를 시작합니다.")

        for m_name, m_data in processed_data.items():
            if m_name in data:
                processed_reviews_dict = {r.get("review_id"): r for r in m_data.get("reviews", [])}
                for i, rev in enumerate(data[m_name].get("reviews", [])):
                    if rev.get("review_id") in processed_reviews_dict:
                        data[m_name]["reviews"][i] = processed_reviews_dict[rev["review_id"]]

    pending_reviews = []
    for m_name, m_data in data.items():
        for rev in m_data.get("reviews", []):
            if rev.get("meta_tags", {}).get("analysis_method") not in ["Local-LLM", "Rule-based (Fallback)"]:
                pending_reviews.append(rev)

    if not pending_reviews:
        print("✅ 모든 리뷰에 대한 분석이 이미 완료되었습니다.")
    else:
        if IS_TEST_MODE:
            pending_reviews = pending_reviews[:TEST_LIMIT]
            print(f"⚠️ [테스트 모드 활성화] 리뷰 수를 {len(pending_reviews)}개로 제한하여 분석합니다.")

        print(f"🚀 총 {len(pending_reviews)}개의 리뷰를 로컬 LLM({LOCAL_MODEL})으로 분석합니다.")

        completed_count = 0
        start_time = time.time()

        for rev in tqdm(pending_reviews, desc="리뷰 분석 진행률"):
            res = classify_single_review_with_local_llm(rev)

            if res:
                rev["meta_tags"].update({
                    "content_character": res.get("content_character", []),
                    "search_keywords": res.get("search_keywords", []),
                    "is_tmi": "TMI" in res.get("content_character", []),
                    "has_warning": "주의사항" in res.get("content_character", []),
                    "analysis_method": "Local-LLM"
                })
            else:
                tags, keywords = classify_with_rule_base(rev["content"])
                rev["meta_tags"].update({
                    "content_character": tags,
                    "search_keywords": keywords,
                    "analysis_method": "Rule-based (Fallback)"
                })

            completed_count += 1

            if completed_count % 50 == 0:
                import datetime
                elapsed = time.time() - start_time
                eta_seconds = (elapsed / completed_count) * (len(pending_reviews) - completed_count)
                elapsed_str = str(datetime.timedelta(seconds=int(elapsed)))
                eta_str = str(datetime.timedelta(seconds=int(eta_seconds)))
                print(f"📦 분석 중... ({completed_count}/{len(pending_reviews)}) | 소요: {elapsed_str} | 남은시간: {eta_str}")

                filtered_data = {}
                for m_name, m_data in data.items():
                    processed_reviews = [r for r in m_data.get("reviews", []) if r.get("meta_tags", {}).get("analysis_method") in ("Local-LLM", "Rule-based (Fallback)")]
                    if processed_reviews:
                        filtered_data[m_name] = {"movie_metadata": m_data.get("movie_metadata", {}), "reviews": processed_reviews}
                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(filtered_data, f, ensure_ascii=False, indent=4)

        filtered_data = {}
        for m_name, m_data in data.items():
            processed_reviews = [r for r in m_data.get("reviews", []) if r.get("meta_tags", {}).get("analysis_method") in ("Local-LLM", "Rule-based (Fallback)")]
            if processed_reviews:
                filtered_data[m_name] = {"movie_metadata": m_data.get("movie_metadata", {}), "reviews": processed_reviews}

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(filtered_data, f, ensure_ascii=False, indent=4)

        print(f"\n✨ 로컬 LLM 분석 완료! 결과 저장: {output_path}")

📝 기존 작업 결과에서 이어하기를 시작합니다.
🚀 총 485개의 리뷰를 로컬 LLM(gemma4)으로 분석합니다.


리뷰 분석 진행률:   0%|          | 0/485 [00:00<?, ?it/s]

📦 분석 중... (50/485) | 소요: 0:18:38 | 남은시간: 2:42:08
📦 분석 중... (100/485) | 소요: 0:34:34 | 남은시간: 2:13:05
📦 분석 중... (150/485) | 소요: 0:53:19 | 남은시간: 1:59:05
📦 분석 중... (200/485) | 소요: 1:10:57 | 남은시간: 1:41:06
📦 분석 중... (250/485) | 소요: 1:27:46 | 남은시간: 1:22:30
📦 분석 중... (300/485) | 소요: 1:44:31 | 남은시간: 1:04:27
📦 분석 중... (350/485) | 소요: 2:03:19 | 남은시간: 0:47:33
📦 분석 중... (400/485) | 소요: 2:20:34 | 남은시간: 0:29:52
📦 분석 중... (450/485) | 소요: 2:39:52 | 남은시간: 0:12:26

✨ 로컬 LLM 분석 완료! 결과 저장: /content/drive/MyDrive/MovieReview/data/tagged_reviews.json
